In [1]:
import sys, pandas, wrds
print(sys.version)
print("ok")


3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
ok


In [2]:
import wrds
db = wrds.Connection(wrds_username="jannatulj")

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\janna\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
print([l for l in db.list_libraries() if l in ("crsp", "comp", "ff")])
print(db.raw_sql("select max(date) from crsp.msf"))


['comp', 'crsp', 'ff']
          max
0  2024-12-31


In [8]:
import pandas as pd
import os
p = "data/raw/kpss/unzipped/"
kp = pd.read_csv(p + "KPSS_2025.csv")
kp["issue_date"] = pd.to_datetime(kp["issue_date"], format="%m/%d/%Y", errors="coerce")
kp["year"] = kp["issue_date"].dt.year

print(kp.shape, kp["issue_date"].min(), kp["issue_date"].max())
print("duplicate patent_num:", kp["patent_num"].duplicated().sum())
print("missing permno:", kp["permno"].isna().mean())

cov = kp.groupby("year").agg(
    patents=("patent_num", "count"),
    firms=("permno", "nunique"),
    mean_xi_real=("xi_real", "mean"),
)
print(cov.loc[2010:2025])
os.makedirs("reports", exist_ok=True)
cov.to_csv("reports/coverage_by_year.csv", index=True)


(3831908, 8) NaT NaT
duplicate patent_num: 0
missing permno: 0.0
Empty DataFrame
Columns: [patents, firms, mean_xi_real]
Index: []


In [ ]:
import os
import pandas as pd
for d in ["reports", "data/interim", "data/processed", "tests", "src"]:
    os.makedirs(d, exist_ok=True)

p = "data/raw/kpss/unzipped/"
if "kp" not in globals():
    kp = pd.read_csv(p + "KPSS_2025.csv")

kp["issue_date"] = pd.to_datetime(kp["issue_date"], format="%m/%d/%Y", errors="coerce")
kp["year"] = kp["issue_date"].dt.year
cov = kp.groupby("year").agg(
    patents=("patent_num", "count"),
    firms=("permno", "nunique"),
    mean_xi_real=("xi_real", "mean"),
)
cov.to_csv("reports/coverage_by_year.csv", index=True)
print("saved")

m = pd.read_csv(p + "Match_patent_permco_permno_2025.csv")
print(m.shape)
print("patents with >1 permno:", m["patent_num"].duplicated().sum())
print("KPSS patents missing from match file:", (~kp["patent_num"].isin(m["patent_num"])).sum())

chk_link = kp[["patent_num", "permno"]].merge(
    m[["patent_num", "permno"]], on="patent_num", suffixes=("_kp", "_match")
)
print("permno disagreements:", (chk_link["permno_kp"] != chk_link["permno_match"]).sum())

if "kp" not in globals():
    kp = pd.read_csv("data/raw/kpss/unzipped/KPSS_2025.csv")

chk_link = kp[["patent_num", "permno"]].merge(
    m[["patent_num", "permno"]], on="patent_num", suffixes=("_kp", "_match")
)
print("permno disagreements:", (chk_link["permno_kp"] != chk_link["permno_match"]).sum())


saved
(3831908, 3)
patents with >1 permno: 0
KPSS patents missing from match file: 0
permno disagreements: 0
permno disagreements: 0


In [8]:
pilot = kp.dropna(subset=["permno"]).copy()
pilot["issue_date"] = pd.to_datetime(pilot["issue_date"], errors="coerce")
pilot["year"] = pilot["issue_date"].dt.year
pilot = pilot.query("year >= 2018")

if pilot.empty:
    raise ValueError("No rows remain after filtering to permno != null and year >= 2018.")

pilot = pilot.sample(100, random_state=42)
permnos = tuple(int(x) for x in pilot["permno"].unique())

ret = db.raw_sql(
    """
    select permno, date, ret, prc, shrout
    from crsp.msf
    where permno in %(p)
      and date between '2017-12-01' and '2024-12-31'
    """,
    params={"p": permnos},
    date_cols=["date"],
)
print(ret.shape, ret["permno"].nunique())

pilot["first_tradable"] = pilot["issue_date"] + pd.offsets.MonthEnd(1)
ret["month_end"] = ret["date"] + pd.offsets.MonthEnd(0)
chk = pilot.merge(
    ret[["permno", "month_end", "ret"]],
    left_on=["permno", "first_tradable"], right_on=["permno", "month_end"],
    how="left",
)
print("share with a return in first tradable month:", chk["ret"].notna().mean())
chk.to_csv("data/interim/pilot_100.csv", index=False)


ValueError: No rows remain after filtering to permno != null and year >= 2018.